# 🏠 Análisis Exploratorio de Datos
## Alquileres de Bienes Inmuebles en Ecuador

**PoliSistemas — Proceso de Selección Técnico de Investigación**  
**Dataset:** `real_state_ecuador_dataset.csv`  
**Objetivo:** Analizar el dataset de alquileres para entender la distribución de precios,
características de los inmuebles, y preparar los datos para modelado predictivo.


## 0. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

print('Librerías cargadas correctamente.')

## 1. Carga del Dataset

In [ ]:
# Carga del dataset
df = pd.read_csv('data/real_state_ecuador_dataset.csv')

print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

In [ ]:
# Información general del dataset
df.info()

## 2. Limpieza y Normalización de Datos

### 2.1 Renombrado de columnas

In [ ]:
# Renombrar columnas para mayor consistencia
df.columns = ['titulo', 'precio', 'provincia', 'lugar_raw', 'num_dormitorios', 'num_banos', 'area', 'num_garages']
print('Columnas renombradas:', df.columns.tolist())

### 2.2 Valores faltantes

In [ ]:
# Análisis de valores faltantes
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos})
print('=== Valores Faltantes ===')
print(resumen_nulos)

# Verificar también NaN en columnas numéricas (pueden estar como string)
for col in ['num_dormitorios', 'num_banos', 'area', 'num_garages']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('\n=== Valores NaN en columnas numéricas (tras conversión) ===')
print(df[['num_dormitorios', 'num_banos', 'area', 'num_garages']].isnull().sum())

In [ ]:
# Imputación de valores faltantes numéricos con la mediana por provincia
cols_numericas = ['num_dormitorios', 'num_banos', 'area', 'num_garages']

for col in cols_numericas:
    if df[col].isnull().sum() > 0:
        mediana_provincia = df.groupby('provincia')[col].transform('median')
        mediana_global = df[col].median()
        df[col] = df[col].fillna(mediana_provincia).fillna(mediana_global)
        print(f'  [{col}] NaN imputados con mediana por provincia')

print('\nTras imputación:')
print(df[cols_numericas].isnull().sum())

### 2.3 Normalización de la columna `Lugar`

La columna `lugar_raw` contiene strings de dirección completa como:
> `"Pichincha, Iñaquito, Quito, Ecuador"`

Se extraerá el nombre de la **ciudad principal** usando un algoritmo de limpieza por reglas.

In [ ]:
def limpiar_lugar(lugar_raw: str, provincia: str) -> str:
    """
    Extrae el nombre de ciudad/cantón desde una dirección completa.
    
    Estrategia:
    1. Dividir por coma
    2. Eliminar 'Ecuador' y términos vacíos
    3. Eliminar la provincia (primer elemento)
    4. Eliminar códigos postales numéricos y plus-codes (ej: VGW2+Q5J)
    5. Eliminar textos que contienen números de código postal inline
    6. Tomar el último elemento significativo restante (la ciudad)
    """
    if pd.isna(lugar_raw):
        return provincia
    
    partes = [p.strip() for p in str(lugar_raw).split(',')]
    
    # Filtros secuenciales
    partes_limpias = []
    for p in partes:
        if not p:
            continue
        if p.lower() == 'ecuador':
            continue
        # Eliminar plus codes (ej: VGW2+Q5J)
        if re.match(r'^[A-Z0-9]{4}\+[A-Z0-9]+', p):
            continue
        # Eliminar si es solo dígitos (código postal)
        if re.match(r'^\d+$', p):
            continue
        partes_limpias.append(p)
    
    if not partes_limpias:
        return provincia
    
    # Quitar primer elemento si coincide con la provincia
    if partes_limpias[0].lower().replace(' ', '') == provincia.lower().replace(' ', ''):
        partes_limpias = partes_limpias[1:]
    
    if not partes_limpias:
        return provincia
    
    # El último elemento suele ser la ciudad principal
    ciudad = partes_limpias[-1]
    
    # Eliminar código postal inline (ej: "Quito 170135" → "Quito")
    ciudad = re.sub(r'\s+\d{5,6}\b', '', ciudad).strip()
    
    return ciudad if ciudad else provincia


# Aplicar limpieza
df['lugar'] = df.apply(lambda row: limpiar_lugar(row['lugar_raw'], row['provincia']), axis=1)

# Mostrar comparación antes/después
muestra = df[['lugar_raw', 'lugar']].sample(10, random_state=42)
print('=== Muestra de Normalización de Lugar ===')
for _, row in muestra.iterrows():
    print(f'  ANTES: {row["lugar_raw"]}')
    print(f'  DESPUÉS: {row["lugar"]}\n')

In [ ]:
# Valores únicos de lugar después de la normalización
print(f'Lugares únicos: {df["lugar"].nunique()}')
print(df['lugar'].value_counts().head(20))

### 2.4 Detección y manejo de outliers en Precio

In [ ]:
# Análisis de outliers en precio usando IQR
Q1 = df['precio'].quantile(0.25)
Q3 = df['precio'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers = df[(df['precio'] < limite_inf) | (df['precio'] > limite_sup)]
print(f'Rango intercuartílico: Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}')
print(f'Límites IQR: [{limite_inf:.0f}, {limite_sup:.0f}]')
print(f'Outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')
print(f'\nDistribución de precio:')
print(df['precio'].describe())

# Se conservan outliers pero se documenta para el modelo
print('\n⚠️ Los outliers se mantienen. Se usará modelo robusto para manejarlos.')

In [ ]:
# Guardar dataset limpio para el modelo
df.to_csv('data/dataset_limpio.csv', index=False)
print('Dataset limpio guardado en data/dataset_limpio.csv')

## 3. Análisis Descriptivo

### 3.1 Conteo de propiedades

In [ ]:
total = len(df)
print(f'Total de propiedades: {total}\n')

# Por Provincia
por_provincia = df.groupby('provincia').size().sort_values(ascending=False).reset_index(name='total')
por_provincia['porcentaje'] = (por_provincia['total'] / total * 100).round(1)
print('=== Propiedades por Provincia ===')
print(por_provincia.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras por provincia
axes[0].barh(por_provincia['provincia'], por_provincia['total'], color=sns.color_palette('muted', len(por_provincia)))
axes[0].set_title('Propiedades por Provincia', fontweight='bold', pad=12)
axes[0].set_xlabel('Cantidad de propiedades')
for i, v in enumerate(por_provincia['total']):
    axes[0].text(v + 0.5, i, f'{v} ({por_provincia.iloc[i]["porcentaje"]}%)', va='center', fontsize=9)

# Pie chart
axes[1].pie(por_provincia['total'], labels=por_provincia['provincia'],
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('pastel', len(por_provincia)))
axes[1].set_title('Distribución por Provincia', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('data/fig_propiedades_provincia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Por Lugar (Top 20)
por_lugar = df.groupby('lugar').size().sort_values(ascending=False).reset_index(name='total')
print(f'\n=== Top 20 Lugares con más propiedades ===' )
print(por_lugar.head(20).to_string(index=False))

# Visualización Top 15
top15 = por_lugar.head(15)
plt.figure(figsize=(14, 7))
bars = plt.barh(top15['lugar'][::-1], top15['total'][::-1],
                color=sns.color_palette('Blues_d', 15))
plt.title('Top 15 Lugares con Mayor Número de Propiedades en Alquiler', fontweight='bold', pad=12)
plt.xlabel('Cantidad de propiedades')
for bar, val in zip(bars, top15['total'][::-1]):
    plt.text(val + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('data/fig_propiedades_lugar.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Estadísticas de precio

In [ ]:
# Estadísticas generales de precio
print('=== Estadísticas Generales de Precio (USD) ===')
stats_precio = df['precio'].agg(['mean', 'median', 'std', 'min', 'max', 
                                   lambda x: x.quantile(0.25), 
                                   lambda x: x.quantile(0.75)])
stats_precio.index = ['Media', 'Mediana', 'Desv. Estándar', 'Mínimo', 'Máximo', 'Q1 (25%)', 'Q3 (75%)']
print(stats_precio.apply(lambda x: f'${x:,.1f}'))

# Histograma de distribución de precios
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma normal
axes[0].hist(df['precio'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['precio'].mean(), color='red', linestyle='--', label=f'Media: ${df["precio"].mean():.0f}')
axes[0].axvline(df['precio'].median(), color='green', linestyle='--', label=f'Mediana: ${df["precio"].median():.0f}')
axes[0].set_title('Distribución de Precios de Alquiler', fontweight='bold')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Log scale para mejor visualización
axes[1].hist(np.log1p(df['precio']), bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribución de Precios (Escala Logarítmica)', fontweight='bold')
axes[1].set_xlabel('log(1 + Precio)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('data/fig_distribucion_precio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Precio promedio y mediana por Lugar (Top 15 lugares con más datos)
top_lugares = por_lugar.head(15)['lugar'].tolist()
stats_por_lugar = (
    df[df['lugar'].isin(top_lugares)]
    .groupby('lugar')['precio']
    .agg(count='count', media='mean', mediana='median', std='std')
    .sort_values('media', ascending=False)
    .reset_index()
)

print('=== Estadísticas de Precio por Lugar (Top 15) ===')
print(stats_por_lugar.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Media
axes[0].barh(stats_por_lugar['lugar'][::-1], stats_por_lugar['media'][::-1],
             color=sns.color_palette('RdYlGn', len(stats_por_lugar))[::-1])
axes[0].set_title('Precio Promedio por Lugar (USD)', fontweight='bold')
axes[0].set_xlabel('Precio promedio (USD)')
for i, v in enumerate(stats_por_lugar['media'][::-1]):
    axes[0].text(v + 5, i, f'${v:.0f}', va='center', fontsize=8)

# Mediana
axes[1].barh(stats_por_lugar['lugar'][::-1], stats_por_lugar['mediana'][::-1],
             color=sns.color_palette('Blues_d', len(stats_por_lugar))[::-1])
axes[1].set_title('Precio Mediano por Lugar (USD)', fontweight='bold')
axes[1].set_xlabel('Precio mediano (USD)')
for i, v in enumerate(stats_por_lugar['mediana'][::-1]):
    axes[1].text(v + 5, i, f'${v:.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('data/fig_precio_por_lugar.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Análisis de la relación entre Área y Precio

In [ ]:
# Correlación Área - Precio
corr = df['area'].corr(df['precio'])
print(f'Correlación de Pearson (Área vs Precio): r = {corr:.3f}')

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter principal
axes[0].scatter(df['area'], df['precio'], alpha=0.4, color='steelblue', edgecolors='none', s=50)
# Línea de tendencia
z = np.polyfit(df['area'].fillna(0), df['precio'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['area'].min(), df['area'].percentile(0.95) if hasattr(df['area'], 'percentile') else df['area'].quantile(0.95), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Tendencia lineal (r={corr:.2f})')
axes[0].set_title('Área vs Precio de Alquiler', fontweight='bold')
axes[0].set_xlabel('Área (m²)')
axes[0].set_ylabel('Precio (USD)')
axes[0].legend()
axes[0].set_xlim(0, df['area'].quantile(0.99))
axes[0].set_ylim(0, df['precio'].quantile(0.99))

# Scatter por provincia con color
provincias_unicas = df['provincia'].unique()
colores = sns.color_palette('tab10', len(provincias_unicas))
for i, prov in enumerate(provincias_unicas):
    mask = df['provincia'] == prov
    axes[1].scatter(df.loc[mask, 'area'], df.loc[mask, 'precio'],
                    alpha=0.5, color=colores[i], label=prov, s=40, edgecolors='none')
axes[1].set_title('Área vs Precio por Provincia', fontweight='bold')
axes[1].set_xlabel('Área (m²)')
axes[1].set_ylabel('Precio (USD)')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].set_xlim(0, df['area'].quantile(0.99))
axes[1].set_ylim(0, df['precio'].quantile(0.99))

plt.tight_layout()
plt.savefig('data/fig_area_precio.png', dpi=150, bbox_inches='tight')
plt.show()

# Precio por m² (eficiencia)
df['precio_m2'] = df['precio'] / df['area']
print(f'\nPrecio por m² - Media: ${df["precio_m2"].mean():.2f} | Mediana: ${df["precio_m2"].median():.2f}')

### 3.4 Premium por Habitación Adicional

In [ ]:
# Premium por habitación adicional (diferencia de precio promedio entre N y N+1 dormitorios)
df['num_dormitorios_int'] = df['num_dormitorios'].round().astype(int)
precio_por_dorm = (
    df.groupby('num_dormitorios_int')['precio']
    .agg(count='count', precio_medio='mean', precio_mediana='median')
    .reset_index()
    .sort_values('num_dormitorios_int')
)

# Calcular premium
precio_por_dorm['premium_vs_anterior'] = precio_por_dorm['precio_medio'].diff()
precio_por_dorm['premium_pct'] = (precio_por_dorm['premium_vs_anterior'] / precio_por_dorm['precio_medio'].shift(1) * 100).round(1)

print('=== Premium por Habitación Adicional ===')
print(precio_por_dorm.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Precio promedio por número de dormitorios
axes[0].bar(precio_por_dorm['num_dormitorios_int'], precio_por_dorm['precio_medio'],
            color=sns.color_palette('viridis', len(precio_por_dorm)))
axes[0].set_title('Precio Promedio por Número de Dormitorios', fontweight='bold')
axes[0].set_xlabel('Número de dormitorios')
axes[0].set_ylabel('Precio promedio (USD)')
for i, row in precio_por_dorm.iterrows():
    axes[0].text(row['num_dormitorios_int'], row['precio_medio'] + 10,
                 f'${row["precio_medio"]:.0f}\n(n={row["count"]})',
                 ha='center', fontsize=9)

# Premium
premium_valido = precio_por_dorm.dropna(subset=['premium_vs_anterior'])
colores_premium = ['red' if v < 0 else 'green' for v in premium_valido['premium_vs_anterior']]
axes[1].bar(
    [f'{int(r.num_dormitorios_int-1)}→{int(r.num_dormitorios_int)}' for _, r in premium_valido.iterrows()],
    premium_valido['premium_vs_anterior'],
    color=colores_premium, alpha=0.8
)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Premium por Habitación Adicional (USD)', fontweight='bold')
axes[1].set_xlabel('Transición de dormitorios')
axes[1].set_ylabel('Diferencia de precio promedio (USD)')
for bar, (_, row) in zip(axes[1].patches, premium_valido.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (5 if bar.get_height() >= 0 else -20),
                 f'+${row["premium_vs_anterior"]:.0f}' if row['premium_vs_anterior'] >= 0 else f'${row["premium_vs_anterior"]:.0f}',
                 ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/fig_premium_habitacion.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Análisis adicionales

In [ ]:
# Matriz de correlación entre variables numéricas
vars_numericas = ['precio', 'num_dormitorios', 'num_banos', 'area', 'num_garages']
corr_matrix = df[vars_numericas].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5,
    mask=mask,
    annot_kws={'size': 12}
)
plt.title('Matriz de Correlación — Variables Numéricas', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('data/fig_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlaciones con Precio:')
print(corr_matrix['precio'].drop('precio').sort_values(ascending=False))

In [ ]:
# Distribución de precio por provincia (boxplot)
plt.figure(figsize=(14, 7))
orden = df.groupby('provincia')['precio'].median().sort_values(ascending=False).index
sns.boxplot(
    data=df[df['precio'] <= df['precio'].quantile(0.95)],  # Excluir extremos para visualización
    x='provincia', y='precio', order=orden,
    palette='Set2'
)
plt.title('Distribución de Precio por Provincia (sin outliers extremos)', fontweight='bold', pad=12)
plt.xlabel('Provincia')
plt.ylabel('Precio de alquiler (USD)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('data/fig_precio_provincia_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis precio vs número de baños
precio_banos = (
    df.groupby(df['num_banos'].round().astype(int))['precio']
    .agg(count='count', media='mean', mediana='median')
    .reset_index()
    .rename(columns={'num_banos': 'num_banos'})
)
print('=== Precio por Número de Baños ===')
print(precio_banos.to_string(index=False))

# Precio vs garages
precio_garages = (
    df.groupby(df['num_garages'].round().astype(int))['precio']
    .agg(count='count', media='mean', mediana='median')
    .reset_index()
)
print('\n=== Precio por Número de Garajes ===')
print(precio_garages.to_string(index=False))

## 4. Creación de Variable: Tipo de Precio por Lugar

Se clasifica cada propiedad según el precio en relación a los cuartiles Q1 y Q3 **calculados por lugar**.

In [ ]:
def clasificar_tipo_precio(grupo: pd.DataFrame) -> pd.Series:
    """
    Clasifica el precio de cada propiedad dentro de su lugar:
    - Económico: precio < Q1
    - Lujo: precio > Q3
    - Medio: resto
    """
    q1 = grupo['precio'].quantile(0.25)
    q3 = grupo['precio'].quantile(0.75)
    
    condiciones = [
        grupo['precio'] < q1,
        grupo['precio'] > q3
    ]
    valores = ['Económico', 'Lujo']
    return pd.Series(np.select(condiciones, valores, default='Medio'), index=grupo.index)


# Aplicar clasificación por lugar
df['tipo_precio_lugar'] = df.groupby('lugar', group_keys=False).apply(clasificar_tipo_precio)

print('=== Distribución de Tipo de Precio por Lugar ===')
dist_tipo = df['tipo_precio_lugar'].value_counts()
print(dist_tipo)
print(f'\nPorcentajes:')
print((dist_tipo / len(df) * 100).round(1))

In [ ]:
# Verificación: ejemplo de clasificación para Quito
quito = df[df['lugar'] == 'Quito'][['precio', 'tipo_precio_lugar']].copy()
q1_q = quito['precio'].quantile(0.25)
q3_q = quito['precio'].quantile(0.75)
print(f'Quito — Q1: ${q1_q:.0f} | Q3: ${q3_q:.0f}')
print(quito['tipo_precio_lugar'].value_counts())

# Visualización de la clasificación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart global
colores_tipo = {'Económico': '#2196F3', 'Medio': '#FF9800', 'Lujo': '#9C27B0'}
conteos = df['tipo_precio_lugar'].value_counts()
axes[0].pie(conteos.values, labels=conteos.index,
            colors=[colores_tipo[t] for t in conteos.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribución Global de Tipo de Precio', fontweight='bold')

# Stacked bar por provincia
tipo_provincia = df.groupby(['provincia', 'tipo_precio_lugar']).size().unstack(fill_value=0)
tipo_provincia_pct = tipo_provincia.div(tipo_provincia.sum(axis=1), axis=0) * 100
orden_cols = ['Económico', 'Medio', 'Lujo']
tipo_provincia_pct = tipo_provincia_pct.reindex(columns=[c for c in orden_cols if c in tipo_provincia_pct.columns])
tipo_provincia_pct.plot(kind='bar', stacked=True, ax=axes[1],
                         color=[colores_tipo[c] for c in tipo_provincia_pct.columns],
                         width=0.7)
axes[1].set_title('Tipo de Precio por Provincia (%)', fontweight='bold')
axes[1].set_xlabel('Provincia')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Tipo de Precio', loc='upper right')

plt.tight_layout()
plt.savefig('data/fig_tipo_precio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Guardar dataset final con tipo_precio_lugar
df.to_csv('data/dataset_limpio.csv', index=False)
print(f'Dataset final guardado. Forma: {df.shape}')
print('Columnas finales:', df.columns.tolist())
df[['titulo', 'precio', 'provincia', 'lugar', 'num_dormitorios', 'num_banos', 'area', 'num_garages', 'tipo_precio_lugar']].head(10)

## 5. Resumen del Análisis Exploratorio

| Hallazgo | Detalle |
|---|---|
| **Total de propiedades** | 500 registros sin valores faltantes iniciales |
| **Provincias** | 10 provincias · Pichincha domina (>60%) |
| **Rango de precio** | $9 – $9,000 USD/mes |
| **Precio medio** | ~$834 USD/mes |
| **Precio mediano** | ~$600 USD/mes (distribución sesgada a la derecha) |
| **Correlación área-precio** | Positiva moderada-alta |
| **Premium por habitación** | El mayor salto es de 1 a 2 dormitorios |
| **Limpieza de Lugar** | Extraída ciudad principal de strings de dirección completa |
| **Tipo de Precio** | Clasificación relativa por lugar (Q1/Q3) |

> **Próximo paso:** Modelado de Machine Learning para predicción de precio → `2_Modelado.ipynb`